# Package

In [ ]:
import importlib

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} is not installed. Installing via !pip ...")
        try:
            get_ipython().system(f"pip install {package_name}")
            print(f"{package_name} installed successfully.")
        except Exception as e:
            print(f"Failed to install {package_name}. Error:\n{e}")

packages = ["pyDOE2", "diversipy", "pygmo", "optproblems", "pymoo"]

for pkg in packages:
    ensure_package(pkg)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyDOE2 import lhs
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import time
import sys
import os
sys.path.append(os.path.abspath('xxx'))

from desdeo_problem.Problem import DataProblem
from desdeo_problem.surrogatemodels.SurrogateKriging import SurrogateKriging
from desdeo_problem.testproblems.TestProblems import test_problem_builder
from desdeo_emo.EAs.ProbRVEA import ProbRVEA_v3
from pymoo.problems import get_problem
from pymoo.operators.sampling.lhs import LHS
from pymoo.problems.multi.omnitest import OmniTest

# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Problem

In [ ]:
def mean_std(arr):
    return np.mean(arr), np.std(arr)

# Initial settings
# Problem: dtlz1-7, omnitest

problem_name = 'dtlz1'
problem_testbench = 'dtlz'

if 'dtlz' in problem_name:
  nvars = 10
  nobjs = 2
  problem_pymoo = get_problem(problem_name, n_var=nvars, n_obj=nobjs)

elif 'omnitest' in problem_name:
  nvars = 2
  nobjs = 2
  problem_pymoo = OmniTest(n_var=nvars)

#nsamples = 11*nvars-1
nsamples = 1000

print(f"Problem name: {problem_name}")
print(f"Cons: {problem_pymoo.n_constr}")
print(f"Var: {nvars}")
print(f"Obj: {nobjs}")


# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

# Other

In [ ]:
# Read dataset
np.random.seed(42)
sampling_pymoo = LHS()
x = sampling_pymoo(problem_pymoo, nsamples, seed=42).get("X")
y = problem_pymoo.evaluate(x, return_values_of=["F"])

print('problem_name', problem_name)
print('x', x.shape)
print('y', y.shape)

In [ ]:
def to_matlab_matrix(X):
    rows = []
    for row in X:
        rows.append(" ".join(f"{v:.4f}" for v in row))
    return "[{}]".format(";\n ".join(rows))

print(to_matlab_matrix(x))


In [ ]:
# Create problem object
is_data = True
x_names = [f'x{i}' for i in range(1,nvars+1)]
y_names = [f'f{i}' for i in range(1,nobjs+1)]
row_names = ['lower_bound','upper_bound']
if is_data is False:
    prob = test_problem_builder(problem_name, nvars, nobjs)

    x = lhs(nvars, nsamples)
    y = prob.evaluate(x)
    data = pd.DataFrame(np.hstack((x,y.objectives)), columns=x_names+y_names)
else:
    data = pd.DataFrame(np.hstack((x,y)), columns=x_names+y_names)
if problem_testbench == 'DDMOPP':
    x_low = np.ones(nvars)*-1
    x_high = np.ones(nvars)
else:
    x_low = problem_pymoo.xl
    x_high = problem_pymoo.xu
    print('x_low: ', x_low)
    print('x_high: ', x_high)


bounds = pd.DataFrame(np.vstack((x_low,x_high)), columns=x_names, index=row_names)
problem = DataProblem(data=data, variable_names=x_names, objective_names=y_names,bounds=bounds)
start = time.time()
problem.train(SurrogateKriging)
end = time.time()
time_taken = end - start
print("Kriging surrogates built in:",time_taken,"(secs)")

# Probabilstic RVEA

In [ ]:
mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, nvars), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, :nobjs-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem_pymoo.evaluate(X_opt)
else:
    pf = problem_pymoo.pareto_front()
igd_plus = IGDPlus(pf)


for seed in range(1,31):
    # Run Probabilstic RVEA (fast)

    start_time = time.time()
    evolver_opt = ProbRVEA_v3(problem,
                              use_surrogates=True,
                              population_size = 100,
                              total_function_evaluations=10000)
    while evolver_opt.continue_evolution():
            evolver_opt.iterate()
    end_time = time.time()

    # Results
    solution = evolver_opt.population.individuals
    obj = evolver_opt.population.objectives
    f_real = problem_pymoo.evaluate(solution, return_values_of=["F"])
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    print(f"\nSeed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"IGD+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f}")


Indicator results

In [ ]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== Prob-RVEA: Statistics over 30 runs ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

In [ ]:
print(problem_name+'_MSE_Prob-RVEA'+' =',mse_list)
print(problem_name+'_IGD+_Prob-RVEA'+' =',igd_list)
print(problem_name+'_HV_Prob-RVEA'+' =',hv_real_list)